<a href="https://colab.research.google.com/github/Rodrigo050705/desafio-infosimples/blob/main/desafio_infosimples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# Bibliotecas que nós instalamos manualmente
!pip install requests beautifulsoup4
from bs4 import BeautifulSoup
import requests

# Bibliotecas nativas do Python
import json

# URL do site
url = 'https://infosimples.com/vagas/desafio/stellarcraft/product.html'

# Objeto contendo a resposta final
resposta_final = {}

# Faz o request
response = requests.get(url)

# Parse do responses
parsed_html = BeautifulSoup(response.content, 'html.parser')

# Vamos pegar o título do produto, na tag H1, com ID "product_title"
resposta_final['title'] = parsed_html.select_one('h1#product_title').get_text()

# Aqui você adiciona os outros campos...

# Vamos pegar a marca do produto, na div, com CLASS "product-brand"
resposta_final['brand'] = parsed_html.select_one('div.product-brand').get_text()

# Vamos pegar as categorias do produto, na div, com CLASS "breadcrumb-bar"
categorias = parsed_html.select_one('div.breadcrumb-bar')
elementos_categorias = categorias.find_all(['a'])
categories = [el.get_text(strip=True) for el in elementos_categorias]
resposta_final['categories'] = categories

# Vamos pegar a descrição do produto, na div, com ID "tab-description"
descricao = parsed_html.select_one('div#tab-description').get_text().strip()
#descricao_formatada = descricao.replace('\n', '')
resposta_final['description'] = descricao

# Vamos pegar a lista com detalhes de cada uma das variações do produto, na div, com CLASS "variant-list"
listas_variantes = parsed_html.select_one('div.variant-list')
variants_array = []
def limpa_preco(preco_str):
    if not preco_str:
        return None
    try:
        # 1. Remove o "R$" e espaços em branco nas pontas
        valor_limpo = preco_str.replace("R$", "").strip()
        # 2. Remove os pontos separadores de milhar
        valor_limpo = valor_limpo.replace(".", "")
        # 3. Substitui a vírgula decimal por ponto (padrão americano do float)
        valor_limpo = valor_limpo.replace(",", ".")
        # 4. Converte para float
        return float(valor_limpo)
    except ValueError:
        return None

for variant_div in listas_variantes.find_all('div', class_=lambda x: x and 'variant-btn' in x):

    # Extrai o Name
    name_el = variant_div.find('div', class_='vname')
    name = name_el.get_text(strip=True) if name_el else None

    # Extrai o Preço Atual
    current_price_el = variant_div.find('div', class_='vprice')
    current_price_raw = current_price_el.get_text(strip=True) if current_price_el else None
    current_price = limpa_preco(current_price_raw)

    # Extrai o Preço Antigo
    old_price_el = variant_div.find('div', class_='vprice-old')
    old_price_raw = old_price_el.get_text(strip=True) if old_price_el else None
    old_price = limpa_preco(old_price_raw)

    # Define a disponibilidade (available)
    # Se a classe da div contiver "unavailable" ou se encontrarmos a div "vunavail", active = False
    classes_da_div = variant_div.get('class', [])
    is_unavailable_by_class = 'unavailable' in classes_da_div
    has_unavail_div = variant_div.find('div', class_='vunavail') is not None

    available = not (is_unavailable_by_class or has_unavail_div)

    # 3. Monta o objeto e adiciona na lista
    variants_array.append({
        "name": name,
        "current_price": current_price,
        "old_price": old_price,
        "available": available
    })

resposta_final['skus'] = variants_array

# Vamos pegar a lista com as propriedades do produto, na div, com ID "tab-specs"
especificacao = []
especificacoes = parsed_html.select_one('div#tab-specs')
for row in especificacoes.find_all('tr'):
  columns = row.find_all('td')

  if len(columns) >= 2:
    label_texto = columns[0].get_text(strip=True)
    value_texto = columns[1].get_text(strip=True)
    especificacao_objeto = {
        "label": label_texto,
        "value": value_texto
    }
  especificacao += [especificacao_objeto]

resposta_final['specification'] = especificacao

# Vamos pegar a lista com as avaliações do produto, na div, com ID "tab-reviews"
reviews = []
listas_avals_variantes = []
listas_avals = parsed_html.select_one('div#tab-reviews')
for item in listas_avals.find_all('div', class_='review-card'):
  nome_aval = item.find('div', class_= 'reviewer-name')
  nome = nome_aval.get_text(strip=True) if nome_aval else "Unknown"

  data_aval = item.find('div', class_='reviewer-date')
  data = data_aval.get_text(strip=True) if data_aval else "Unknown"

  score_aval = item.find('div', class_='review-stars')
  formatar_score = score_aval.get_text().count('★')
  score = int(formatar_score)

  texto_aval = item.find('p', class_='review-text')
  texto = texto_aval.get_text(strip=True) if texto_aval else "Unknown"

  listas_avals_variantes.append({
      "name": nome,
      "date": data,
      "score": score,
      "text": texto
  })

reviews = listas_avals_variantes
resposta_final['reviews'] = reviews


# Vamos pegar a nota média das avaliações do produto, na tag span, com CLASS "stars"
nota_media = parsed_html.select_one('span.stars').get_text()
nota_media_formatada = nota_media.replace('★', '')
nota_media_float = float(nota_media_formatada)
resposta_final['reviews_average_score'] = nota_media_float

# Vamos pegar a URL da página do produto
resposta_final['url'] = response.url

# Gera string JSON com a resposta final
json_resposta_final = json.dumps(resposta_final, ensure_ascii=False, indent=4)

# Salva o arquivo JSON com a resposta final
with open('produto.json', 'w') as arquivo_json:
  arquivo_json.write(json_resposta_final)